[Reference](https://blog.dataengineerthings.org/how-to-build-a-medallion-architecture-locally-with-dbt-and-duckdb-e8e5e9270a72)

# Step 1: Extraction & Loading (The “E” and “L”)

In [1]:
import duckdb
import pandas as pd

# Connect to (or create) a local DuckDB file
con = duckdb.connect('coffee_shop.duckdb')

# Load Sales Data (CSV)
con.execute("""
    CREATE TABLE IF NOT EXISTS raw_sales AS
    SELECT * FROM read_csv_auto('data/daily_sales.csv')
""")

# Load Inventory Data (JSON)
con.execute("""
    CREATE TABLE IF NOT EXISTS raw_inventory AS
    SELECT * FROM read_json_auto('data/inventory.json')
""")

print("Successfully loaded raw data into DuckDB!")
con.close()

# Step 2: dbt setup

## 1. Initialize your dbt Project
```
dbt init my_coffee_shop_project
```

## 2. Configure your profiles.yml
```
my_coffee_shop_project:
  outputs:
    dev:
      type: duckdb
      path: 'path/to/your/coffee_shop.duckdb'  # Point this to your local file
      threads: 4
  target: dev
```

## 3. Define your Sources
```
version: 2

sources:
  - name: raw_data
    schema: main # DuckDB defaults to the 'main' schema
    tables:
      - name: raw_sales
      - name: raw_inventory
```

# Step 3: Transformation with dbt (The “T”)
```
models/
├── bronze/
├── silver/
└── gold/
```

## The Bronze Layer
```sql
{{ config(materialized='table') }}

SELECT
    *,
    current_timestamp as _ingested_at
FROM {{ source('raw_data', 'raw_sales') }}
```

## The Silver Layer
```sql
-- models/silver/slv_sales_cleaned.sql

{{ config(materialized='view') }}

WITH base_sales AS (
    SELECT
        CAST(transaction_id AS INT) as transaction_id,
        CAST(sale_date AS DATE) as sale_date,
        product_id,
        CAST(quantity AS INT) as quantity,
        CAST(price AS DECIMAL(10,2)) as price
    FROM {{ ref('brz_sales') }}
    WHERE transaction_id IS NOT NULL
)

SELECT
    s.*,
    (s.quantity * s.price) as total_line_amount
FROM base_sales s
```

## The Gold Layer
```sql
-- models/gold/gld_daily_revenue.sql
{{ config(materialized='table') }}

SELECT
    sale_date,
    SUM(total_line_amount) as daily_revenue,
    count(distinct transaction_id) as total_transactions
FROM {{ ref('slv_sales_cleaned') }}
GROUP BY 1
ORDER BY 1 DESC
```

# Step 4: Executing the Pipeline
```
dbt run
```